In [2]:
pip install pandas openpyxl tokenizers sentencepiece transformers tqdm numpy


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install pandas datasets transformers torch evaluate sentencepiece tokenizers accelerate openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas datasets transformers torch evaluate sentencepiece tokenizers accelerate openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [1]:
# fine tuning nllb. 
#!/usr/bin/env python3
# Fine-tune NLLB-200 for 2 supported low-resource Indic languages (excluding Meitei Mayek).
#Uses all training data. 

import os
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import torch

# ==================== CONFIGURATION ====================
DATA_DIR = "/home/dingku/Desktop/WMT26/WMT26/Cat1/"
MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_BASE_DIR = "/home/dingku/Desktop/finetuned_nllb_indic/"

LANGUAGES = [
    {"name": "Assamese", "code": "asm_Beng", "file": "English-AssameseTrainingData2026.xlsx", "col": "as"},
    {"name": "Manipuri", "code": "mni_Mtei", "file": "English-ManipuriTrainingData2026.xlsx", "col": "mni"},
    #{"name": "Mizo",     "code": "lus_Latn", "file": "English-MizoTraningData2026.xlsx", "col": "mz"},
]

# Hyperparameters
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
EPOCHS = 3
MAX_LENGTH = 128

# ==================== DATA LOADING ====================
def load_and_prepare_data(file_path, lang_code, target_col):
    """Load Excel, clean NaNs, add language prefixes, return Dataset."""
    full_path = os.path.join(DATA_DIR, file_path)
    df = pd.read_excel(full_path)

    # Identify English column
    if 'en' in df.columns:
        eng_col = 'en'
    else:
        eng_candidates = [c for c in df.columns if 'english' in c.lower()]
        eng_col = eng_candidates[0] if eng_candidates else df.columns[0]

    # Rename
    df = df.rename(columns={eng_col: 'en', target_col: 'target'})
    df = df[['en', 'target']].dropna()
    df['en'] = df['en'].astype(str).str.strip()
    df['target'] = df['target'].astype(str).str.strip()
    df = df[(df['en'] != '') & (df['target'] != '')]

    # Add language prefixes
    df['source'] = "eng_Latn " + df['en']
    df['target'] = f"{lang_code} " + df['target']

    print(f"  Loaded {len(df)} valid pairs for {lang_code}")
    return Dataset.from_dict({
        'source': df['source'].tolist(),
        'target': df['target'].tolist()
    })

# ==================== TOKENIZATION ====================
def tokenize_function(examples, tokenizer, lang_code):
    tokenizer.src_lang = "eng_Latn"
    source_inputs = tokenizer(
        examples['source'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    tokenizer.tgt_lang = lang_code
    target_inputs = tokenizer(
        examples['target'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    source_inputs['labels'] = target_inputs['input_ids']
    return source_inputs

# ==================== TRAINING ====================
def train_for_language(lang_config):
    print(f"\n{'='*50}")
    print(f"Training {lang_config['name']} (code: {lang_config['code']})")
    print(f"{'='*50}")

    dataset = load_and_prepare_data(
        lang_config['file'],
        lang_config['code'],
        lang_config['col']
    )

    print("  Loading NLLB model & tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    print("  Tokenizing dataset...")
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x, tokenizer, lang_config['code']),
        batched=True,
        remove_columns=dataset.column_names
    )

    output_dir = os.path.join(OUTPUT_BASE_DIR, lang_config['name'])
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="no",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        num_train_epochs=EPOCHS,
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        logging_dir=os.path.join(output_dir, 'logs'),
        logging_steps=50,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    print("  Starting training...")
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"  Model saved to {output_dir}\n")

# ==================== MAIN ====================
def main():
    os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
    for lang in LANGUAGES:
        try:
            train_for_language(lang)
        except Exception as e:
            print(f"!!! ERROR for {lang['name']}: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    main()

/home/dingku/jupyter_env/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._regist


Training Assamese (code: asm_Beng)
  Loaded 54000 valid pairs for asm_Beng
  Loading NLLB model & tokenizer...


/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/modeling_utils.py:519: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by t

  Tokenizing dataset...


Map:   0%|          | 0/54000 [00:00<?, ? examples/s]

...

  Tokenizing dataset...


Map:   0%|          | 0/49958 [00:00<?, ? examples/s]

/home/dingku/jupyter_env/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
You're using a NllbTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Starting training...
!!! ERROR for Mizo: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 17.38 MiB is free. Including non-PyTorch memory, this process has 15.87 GiB memory in use. Of the allocated memory 15.40 GiB is allocated by PyTorch, and 185.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Traceback (most recent call last):
  File "/tmp/ipykernel_28140/2351792362.py", line 149, in main
    train_for_language(lang)
  File "/tmp/ipykernel_28140/2351792362.py", line 139, in train_for_language
    trainer.train()
  File "/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/trainer.py", line 1537, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/trainer.py", line 1902, in _inner_training_loop
    self.optimizer.step()
  File "/home/dingku/jupyter_env/lib/python3.12/site-packages/accelerate/optimizer.py", line 132, in step
    self.scaler.step(self.optimizer, closure)
  File "/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/amp/grad_scaler.py", line 454, in step
    retval = self._maybe_opt_step(optimizer, optimizer_state, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dingku/jupyter_env/lib/p

In [2]:
#!/usr/bin/env python3
"""
Fine-tune NLLB-200 for Manipuri (mni_Beng) using all training data.
"""

import os
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import torch

# ==================== CONFIGURATION ====================
DATA_DIR = "/home/dingku/Desktop/WMT26/WMT26/Cat1/"
MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = "/home/dingku/Desktop/finetuned_nllb_indic/Manipuri_Beng"

# Manipuri specific
LANG_CODE = "mni_Beng"          # Bengali script
FILE_NAME = "English-ManipuriTrainingData2026.xlsx"
TARGET_COL = "mni"              # column name in Excel

# Hyperparameters
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
EPOCHS = 3
MAX_LENGTH = 128

# ==================== DATA LOADING ====================
def load_and_prepare_data():
    full_path = os.path.join(DATA_DIR, FILE_NAME)
    df = pd.read_excel(full_path)

    # Identify English column
    if 'en' in df.columns:
        eng_col = 'en'
    else:
        eng_candidates = [c for c in df.columns if 'english' in c.lower()]
        eng_col = eng_candidates[0] if eng_candidates else df.columns[0]

    # Rename to standard names
    df = df.rename(columns={eng_col: 'en', TARGET_COL: 'target'})
    df = df[['en', 'target']].dropna()
    df['en'] = df['en'].astype(str).str.strip()
    df['target'] = df['target'].astype(str).str.strip()
    df = df[(df['en'] != '') & (df['target'] != '')]

    # Add language prefixes
    df['source'] = "eng_Latn " + df['en']
    df['target'] = f"{LANG_CODE} " + df['target']

    print(f"Loaded {len(df)} valid pairs for Manipuri ({LANG_CODE})")
    return Dataset.from_dict({
        'source': df['source'].tolist(),
        'target': df['target'].tolist()
    })

# ==================== TOKENIZATION ====================
def tokenize_function(examples, tokenizer):
    tokenizer.src_lang = "eng_Latn"
    source_inputs = tokenizer(
        examples['source'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    tokenizer.tgt_lang = LANG_CODE
    target_inputs = tokenizer(
        examples['target'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    source_inputs['labels'] = target_inputs['input_ids']
    return source_inputs

# ==================== MAIN TRAINING ====================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Load data
    dataset = load_and_prepare_data()

    # 2. Load model and tokenizer
    print("Loading NLLB model & tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    # 3. Tokenize dataset
    print("Tokenizing dataset...")
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=dataset.column_names
    )

    # 4. Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        evaluation_strategy="no",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        num_train_epochs=EPOCHS,
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
        logging_steps=50,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    # 5. Trainer
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    # 6. Train and save
    print("Starting training...")
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Loaded 23687 valid pairs for Manipuri (mni_Beng)
Loading NLLB model & tokenizer...


/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/modeling_utils.py:519: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by t

Tokenizing dataset...


Map:   0%|          | 0/23687 [00:00<?, ? examples/s]

/home/dingku/jupyter_env/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
You're using a NllbTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training...


Step,Training Loss
50,2.167200
100,1.649100
150,1.653600
200,1.532300
250,1.622700
300,1.542100
350,1.612400
400,1.486600
450,1.626600
500,1.404700


Model saved to /home/dingku/Desktop/finetuned_nllb_indic/Manipuri_Beng


In [2]:
#!/usr/bin/env python3
#Real-time translation using fine-tuned NLLB models for Assamese and Manipuri.
#Reads English sentences from Excel files, translates, displays progress, saves results.

import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import time

# ============================================================
# CONFIGURATION
# ============================================================

# Paths to your fine-tuned models
MODEL_PATHS = {
    "asm": "/home/dingku/Desktop/finetuned_nllb_indic/Assamese",
    "mni": "/home/dingku/Desktop/finetuned_nllb_indic/Manipuri_Beng"
}

# Input Excel files
INPUT_FILES = {
    "asm": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Assamese/en-as Test.xlsx",
    "mni": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Manipuri/en-mni Test.xlsx"
}

# Output text files (saved on Desktop)
OUTPUT_FILES = {
    "asm": "/home/dingku/Desktop/translated_assamese.txt",
    "mni": "/home/dingku/Desktop/translated_manipuri.txt"
}

# Column name containing English sentences
EN_COL = "English Sentences"

# Device: use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# LOAD MODELS
# ============================================================

def load_model(lang_code):
    """Load tokenizer and model for given language."""
    model_path = MODEL_PATHS[lang_code]
    print(f"\nLoading {lang_code.upper()} model from {model_path} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    model.to(device)
    model.eval()
    return tokenizer, model

# Load both models
tokenizers = {}
models = {}
for lang in ["asm", "mni"]:
    tok, mdl = load_model(lang)
    tokenizers[lang] = tok
    models[lang] = mdl

# ============================================================
# TRANSLATION FUNCTION
# ============================================================

def translate_sentences(lang_code, sentences, batch_size=8):
    """
    Translate a list of English sentences into the target language.
    Yields (original, translation) one by one for real-time display.
    """
    tokenizer = tokenizers[lang_code]
    model = models[lang_code]
    
    # Process in batches for efficiency, but yield after each sentence
    for i, sent in enumerate(tqdm(sentences, desc=f"Translating {lang_code.upper()}")):
        if not isinstance(sent, str) or not sent.strip():
            yield sent, ""
            continue
        
        # Tokenize
        inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate translation
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        yield sent, translation

# ============================================================
# MAIN TRANSLATION LOOP
# ============================================================

def translate_and_save(lang_code, input_excel, output_txt):
    """Read Excel, translate each sentence, save results, show in real time."""
    print(f"\n{'='*60}")
    print(f"Processing {lang_code.upper()} translation")
    print(f"Input:  {input_excel}")
    print(f"Output: {output_txt}")
    print(f"{'='*60}\n")
    
    # Read Excel
    df = pd.read_excel(input_excel)
    if EN_COL not in df.columns:
        raise ValueError(f"Column '{EN_COL}' not found in {input_excel}")
    
    english_sentences = df[EN_COL].tolist()
    print(f"Total sentences to translate: {len(english_sentences)}")
    
    # Open output file
    with open(output_txt, "w", encoding="utf-8") as f_out:
        f_out.write(f"Translations from English to {lang_code.upper()}\n")
        f_out.write(f"Model: {MODEL_PATHS[lang_code]}\n")
        f_out.write("-" * 80 + "\n\n")
        
        # Translate one by one and display
        for idx, (orig, trans) in enumerate(translate_sentences(lang_code, english_sentences), start=1):
            # Real-time print to console
            print(f"\n[{idx}/{len(english_sentences)}]")
            print(f"EN: {orig}")
            print(f"{lang_code.upper()}: {trans}")
            print("-" * 50)
            
            # Write to file
            f_out.write(f"Original ({idx}): {orig}\n")
            f_out.write(f"Translated: {trans}\n")
            f_out.write("-" * 50 + "\n")
            f_out.flush()  # ensure immediate write
    
    print(f"\n✅ Translation saved to: {output_txt}")

# ============================================================
# RUN TRANSLATIONS
# ============================================================

if __name__ == "__main__":
    # Translate Assamese
    translate_and_save("asm", INPUT_FILES["asm"], OUTPUT_FILES["asm"])
    
    # Translate Manipuri
    translate_and_save("mni", INPUT_FILES["mni"], OUTPUT_FILES["mni"])
    
    print("\n🎉 All translations completed successfully!")

Using device: cuda

Loading ASM model from /home/dingku/Desktop/finetuned_nllb_indic/Assamese ...

Loading MNI model from /home/dingku/Desktop/finetuned_nllb_indic/Manipuri_Beng ...

Processing ASM translation
Input:  /home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Assamese/en-as Test.xlsx
Output: /home/dingku/Desktop/translated_assamese.txt

Total sentences to translate: 1000


Translating ASM:   0%|                         | 1/1000 [00:00<09:12,  1.81it/s]


[1/1000]
EN: NDTV has learnt from sources that the T20 World Cup 2026 match between India and Pakistan is ON.
ASM: এনডিটিভিয়ে সূত্ৰৰ পৰা জানিব পাৰিছে যে ভাৰত আৰু পাকিস্তানৰ মাজত টি২০ বিশ্বকাপ ২০২৬ৰ মেচ অন ।
--------------------------------------------------


Translating ASM:   0%|                         | 2/1000 [00:00<07:13,  2.30it/s]


[2/1000]
EN: The PCB placed several demands before the ICC.
ASM: পিচবিটিয়ে আইচিচিৰ আগত কেইবাটাও দাবী দাখিল কৰিছিল ।
--------------------------------------------------


...


[998/1000]
EN: Oncologist shares case of non-tobacco related cancer in 21-year-old man with ‘sharp teeth,’ says ‘It could have been identified’
MNI: ওঙ্কোলোজিস্তনা চহি ২১ শুরবা শর্প দেন্ত লৈবা নুপা অমদা মৈখু নত্তনা ক্যান্সরগা মরি লৈনবা কেস অমা শন্দোক্না তাক্লি ।
--------------------------------------------------


Translating MNI: 100%|██████████████████████▉| 999/1000 [09:09<00:00,  1.52it/s]


[999/1000]
EN: Nutritionist says pineapple and cinnamon may help reduce period cramps naturally
MNI: ন্যুত্রিসনিষ্টনা হায়, এনিয়াপল অমসুং সিনেমোননা মহৌশাগী মওংদা পেরিওদ ক্রাম্প হন্থহনবদা মতেং পাংই ।
--------------------------------------------------


Translating MNI: 100%|██████████████████████| 1000/1000 [09:10<00:00,  1.82it/s]


[1000/1000]
EN: Don’t brush before bed? Your heart might pay the price
MNI: তুমদ্রিঙৈদা ব্রশ তৌরোইদবনি ।
--------------------------------------------------

✅ Translation saved to: /home/dingku/Desktop/translated_manipuri.txt

🎉 All translations completed successfully!


In [2]:
#Finetuned IT2 1B

In [3]:
pip install transformers datasets peft pandas openpyxl accelerate bitsandbytes

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [3]:
import subprocess, sys

def run(cmd):
    subprocess.check_call(cmd)

# Upgrade pip
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Uninstall conflicting packages
run([sys.executable, "-m", "pip", "uninstall", "-y", "transformers", "tokenizers", "accelerate", "peft"])

# First install a tokenizers version that has a prebuilt wheel for Python 3.12
run([sys.executable, "-m", "pip", "install", "tokenizers==0.15.1", "--only-binary=:all:"])

# Now install transformers 4.36.0 (uses the already installed tokenizers)
run([sys.executable, "-m", "pip", "install", "transformers==4.36.0"])

# Other required packages
run([sys.executable, "-m", "pip", "install", "accelerate==0.25.0"])
run([sys.executable, "-m", "pip", "install", "sentencepiece"])
run([sys.executable, "-m", "pip", "install", "peft==0.7.0"])
run([sys.executable, "-m", "pip", "install", "datasets"])
run([sys.executable, "-m", "pip", "install", "pandas", "openpyxl"])

print("✅ All packages installed. Please restart the kernel (Kernel → Restart) and then run the training script.")

    PyYAML (>=5.1.*)
            ~~~~~~^


Found existing installation: peft 0.4.0
Uninstalling peft-0.4.0:
  Successfully uninstalled peft-0.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.3 MB/s  0:00:00m 5.2 MB/s eta 0:00:01


    PyYAML (>=5.1.*)
            ~~~~~~^


  Using cached transformers-4.36.0-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.0-py3-none-any.whl (8.2 MB)


    PyYAML (>=5.1.*)
            ~~~~~~^


...

    PyYAML (>=5.1.*)
            ~~~~~~^


    PyYAML (>=5.1.*)
            ~~~~~~^


    PyYAML (>=5.1.*)
            ~~~~~~^


✅ All packages installed. Please restart the kernel (Kernel → Restart) and then run the training script.


    PyYAML (>=5.1.*)
            ~~~~~~^


In [3]:
pip install torch transformers datasets pandas openpyxl tqdm peft accelerate

    PyYAML (>=5.1.*)
            ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [4]:
#!/usr/bin/env python3

#Convert Excel training files into the official IndicTrans2 folder structure: en-indic-exp/

import os
import pandas as pd

BASE_DIR = "/home/dingku/Desktop/en-indic-exp"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
DEV_DIR   = os.path.join(BASE_DIR, "dev")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(DEV_DIR, exist_ok=True)

# Map language codes to target folders and column names
LANG_CONFIG = {
    "asm_Beng": {
        "file": "/home/dingku/Desktop/WMT26/WMT26/Cat1/English-AssameseTrainingData2026.xlsx",
        "en_col": "en",
        "tgt_col": "as",
        "folder": "eng_Latn-asm_Beng"
    },
    "mni_Beng": {
        "file": "/home/dingku/Desktop/WMT26/WMT26/Cat1/English-ManipuriTrainingData2026.xlsx",
        "en_col": "en",
        "tgt_col": "mni",
        "folder": "eng_Latn-mni_Beng"
    },
    "mni_Mtei": {
        "file": "/home/dingku/Desktop/WMT26/WMT26/Cat1/English-Meitei-MayekTransiningData2026.xlsx",
        "en_col": "en",
        "tgt_col": "mtei",
        "folder": "eng_Latn-mni_Mtei"
    },
    "brx_Deva": {
        "file": "/home/dingku/Desktop/WMT26/WMT26/Cat2/English-BodoTrainingData2026.xlsx",
        "en_col": "en",
        "tgt_col": "bo",
        "folder": "eng_Latn-brx_Deva"
    }
}

for lang, cfg in LANG_CONFIG.items():
    print(f"Processing {lang} ...")
    df = pd.read_excel(cfg["file"])
    en_sentences = df[cfg["en_col"]].astype(str).tolist()
    tgt_sentences = df[cfg["tgt_col"]].astype(str).tolist()

    # Remove empty rows
    pairs = [(e, t) for e, t in zip(en_sentences, tgt_sentences) if e.strip() and t.strip()]
    print(f"  Total valid pairs: {len(pairs)}")

    # Create folder for this language pair
    folder_path = os.path.join(TRAIN_DIR, cfg["folder"])
    os.makedirs(folder_path, exist_ok=True)

    # Write English sentences (train.eng_Latn)
    en_path = os.path.join(folder_path, "train.eng_Latn")
    with open(en_path, "w", encoding="utf-8") as f:
        for e, _ in pairs:
            f.write(e + "\n")

    # Write target sentences (train.<tgt_lang>)
    tgt_path = os.path.join(folder_path, f"train.{lang}")
    with open(tgt_path, "w", encoding="utf-8") as f:
        for _, t in pairs:
            f.write(t + "\n")

    print(f"  ✅ Saved to {folder_path}")

print("\n🎉 All files created. You can now use the official IndicTrans2 training script.")

Processing asm_Beng ...
  Total valid pairs: 54000
  ✅ Saved to /home/dingku/Desktop/en-indic-exp/train/eng_Latn-asm_Beng
Processing mni_Beng ...
  Total valid pairs: 23687
  ✅ Saved to /home/dingku/Desktop/en-indic-exp/train/eng_Latn-mni_Beng
Processing mni_Mtei ...
  Total valid pairs: 17000
  ✅ Saved to /home/dingku/Desktop/en-indic-exp/train/eng_Latn-mni_Mtei
Processing brx_Deva ...
  Total valid pairs: 15215
  ✅ Saved to /home/dingku/Desktop/en-indic-exp/train/eng_Latn-brx_Deva

🎉 All files created. You can now use the official IndicTrans2 training script.


In [1]:
#!/usr/bin/env python3

#Fine‑tune IndicTrans2 (1.1B) with LoRA for English‑Assamese, English‑Manipuri (Bengali & Meitei Mayek), English‑Bodo.
#Fixed AMP error: use float32 (or fp16) instead of bfloat16.
#Shows real‑time progress and estimated remaining time.

import os
import time
from datetime import timedelta
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
from transformers.trainer_callback import TrainerCallback

# ============================================================
# CONFIGURATION
# ============================================================

DATA_ROOT = "/home/dingku/Desktop/en-indic-exp"
LANG_PAIRS = [
    "eng_Latn-asm_Beng",
    "eng_Latn-mni_Beng",
    "eng_Latn-mni_Mtei",
    "eng_Latn-brx_Deva"
]

BASE_MODEL = "ai4bharat/indictrans2-en-indic-1B"
OUTPUT_BASE = "/home/dingku/Desktop/FTIT26"

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj"]

# Training hyperparameters
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 512
EVAL_STEPS = 500
SAVE_STEPS = 500
LOGGING_STEPS = 50

# Mixed precision: set to True for fp16 (faster), False for float32 (stable)
USE_FP16 = False   # Change to True if your GPU supports fp16 and you want speed
USE_BF16 = False   # Keep False to avoid the scaler error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# CUSTOM CALLBACK FOR REMAINING TIME
# ============================================================
class TimeEstimateCallback(TrainerCallback):
    def __init__(self):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:
            elapsed = time.time() - self.start_time
            steps_done = state.global_step
            if steps_done > 0:
                time_per_step = elapsed / steps_done
                steps_remaining = state.max_steps - steps_done
                eta = timedelta(seconds=int(time_per_step * steps_remaining))
                tqdm.write(
                    f"Step {steps_done}/{state.max_steps}  |  "
                    f"Elapsed: {timedelta(seconds=int(elapsed))}  |  "
                    f"ETA: {eta}"
                )


# ============================================================
# DATA LOADER
# ============================================================
def load_data_for_pair(pair_name):
    train_dir = os.path.join(DATA_ROOT, "train", pair_name)
    dev_dir   = os.path.join(DATA_ROOT, "dev", pair_name)

    src_lang, tgt_lang = pair_name.split("-")

    # Load training files
    with open(os.path.join(train_dir, f"train.{src_lang}"), "r", encoding="utf-8") as f:
        src_lines = [line.strip() for line in f if line.strip()]
    with open(os.path.join(train_dir, f"train.{tgt_lang}"), "r", encoding="utf-8") as f:
        tgt_lines = [line.strip() for line in f if line.strip()]

    assert len(src_lines) == len(tgt_lines), "Mismatch in training data"
    train_data = [{"en": s, "tgt": t} for s, t in zip(src_lines, tgt_lines)]

    # Load or split dev data
    if os.path.exists(dev_dir):
        with open(os.path.join(dev_dir, f"dev.{src_lang}"), "r", encoding="utf-8") as f:
            dev_src = [line.strip() for line in f if line.strip()]
        with open(os.path.join(dev_dir, f"dev.{tgt_lang}"), "r", encoding="utf-8") as f:
            dev_tgt = [line.strip() for line in f if line.strip()]
        dev_data = [{"en": s, "tgt": t} for s, t in zip(dev_src, dev_tgt)]
    else:
        split = int(len(train_data) * 0.9)
        dev_data = train_data[split:]
        train_data = train_data[:split]

    print(f"  Train samples: {len(train_data):,}  |  Dev samples: {len(dev_data):,}")
    return DatasetDict({
        "train": Dataset.from_list(train_data),
        "validation": Dataset.from_list(dev_data)
    })


# ============================================================
# TOKENIZE FUNCTION (FIXED FOR INDICTRANS2)
# ============================================================
def preprocess_function(examples, tokenizer, src_lang, tgt_lang):
    """
    Encoder input:   f"{src_lang} {tgt_lang} {sentence}"
    Decoder input:   f"{tgt_lang} {translation}"  (as text_target)
    """
    inputs = [f"{src_lang} {tgt_lang} {ex}" for ex in examples["en"]]
    targets = [f"{tgt_lang} {ex}" for ex in examples["tgt"]]

    model_inputs = tokenizer(
        inputs, max_length=MAX_LENGTH, truncation=True, padding=False
    )
    # Use text_target argument (modern way, avoids deprecated as_target_tokenizer)
    labels = tokenizer(
        text_target=targets, max_length=MAX_LENGTH, truncation=True, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# ============================================================
# FINE‑TUNING FUNCTION
# ============================================================
def fine_tune_pair(pair_name):
    print("\n" + "=" * 70)
    print(f"🚀 Fine‑tuning for {pair_name}")
    print("=" * 70)

    # 1. Prepare dataset
    dataset = load_data_for_pair(pair_name)

    # 2. Load tokenizer and base model (float32 to avoid bfloat16 scaler issues)
    print("\n📥 Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        use_fast=True
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        torch_dtype=torch.float32   # Use float32 (stable)
    )
    # Enable gradient checkpointing to save memory
    model.gradient_checkpointing_enable()
    model.to(device)

    # 3. Add LoRA adapters
    print("🔧 Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # 4. Tokenize datasets
    print("\n📝 Tokenizing datasets...")
    src_lang, tgt_lang = pair_name.split("-")
    tokenized_ds = dataset.map(
        lambda x: preprocess_function(x, tokenizer, src_lang, tgt_lang),
        batched=True,
        remove_columns=dataset["train"].column_names,
    )

    # 5. Data collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True
    )

    # 6. Training arguments
    output_dir = os.path.join(OUTPUT_BASE, pair_name)
    os.makedirs(output_dir, exist_ok=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_steps=SAVE_STEPS,
        logging_steps=LOGGING_STEPS,
        report_to="none",                # disable wandb / tensorboard
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        fp16=USE_FP16,
        bf16=USE_BF16,
        push_to_hub=False,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    # 7. Trainer with time‑estimate callback
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[TimeEstimateCallback()],
    )

    # 8. Train
    print("\n🏋️ Starting training...\n")
    trainer.train()

    # 9. Save the final LoRA adapter
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n✅ Model saved to {output_dir}\n")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_BASE, exist_ok=True)

    for pair in LANG_PAIRS:
        fine_tune_pair(pair)

    print("\n🎉 All models have been fine‑tuned successfully!")

/home/dingku/jupyter_env/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._regist

Using device: cuda

🚀 Fine‑tuning for eng_Latn-asm_Beng
  Train samples: 48,600  |  Dev samples: 5,400

📥 Loading tokenizer and base model...


/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


🔧 Adding LoRA adapters...
trainable params: 3,538,944 || all params: 1,119,082,496 || trainable%: 0.3162362035550952

📝 Tokenizing datasets...


Map:   0%|          | 0/48600 [00:00<?, ? examples/s]

...

/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step 5050/5736  |  Elapsed: 1:22:34  |  ETA: 0:11:13
Step 5100/5736  |  Elapsed: 1:23:16  |  ETA: 0:10:23
Step 5150/5736  |  Elapsed: 1:23:59  |  ETA: 0:09:33
Step 5200/5736  |  Elapsed: 1:24:41  |  ETA: 0:08:43
Step 5250/5736  |  Elapsed: 1:25:22  |  ETA: 0:07:54
Step 5300/5736  |  Elapsed: 1:26:05  |  ETA: 0:07:04
Step 5350/5736  |  Elapsed: 1:26:47  |  ETA: 0:06:15
Step 5400/5736  |  Elapsed: 1:27:30  |  ETA: 0:05:26
Step 5450/5736  |  Elapsed: 1:28:17  |  ETA: 0:04:38
Step 5500/5736  |  Elapsed: 1:29:01  |  ETA: 0:03:49


/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step 5550/5736  |  Elapsed: 1:30:34  |  ETA: 0:03:02
Step 5600/5736  |  Elapsed: 1:31:18  |  ETA: 0:02:13
Step 5650/5736  |  Elapsed: 1:32:00  |  ETA: 0:01:24
Step 5700/5736  |  Elapsed: 1:32:43  |  ETA: 0:00:35

✅ Model saved to /home/dingku/Desktop/FTIT26/eng_Latn-mni_Mtei


🚀 Fine‑tuning for eng_Latn-brx_Deva


AssertionError: Mismatch in training data

In [1]:
#!/usr/bin/env python3

#Continue fine‑tuning Bodo (eng_Latn-brx_Deva) only.
#Assamese, Manipuri (Bengali), Manipuri (Meitei Mayek) are already done.

import os
import time
from datetime import timedelta
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
from transformers.trainer_callback import TrainerCallback

# ============================================================
# CONFIGURATION – BODO ONLY
# ============================================================

DATA_ROOT = "/home/dingku/Desktop/en-indic-exp"
LANG_PAIRS = ["eng_Latn-brx_Deva"]          # <-- only Bodo

BASE_MODEL = "ai4bharat/indictrans2-en-indic-1B"
OUTPUT_BASE = "/home/dingku/Desktop/FTIT26"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj"]

BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 512
EVAL_STEPS = 500
SAVE_STEPS = 500
LOGGING_STEPS = 50

USE_FP16 = False
USE_BF16 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# TIME ESTIMATE CALLBACK
# ============================================================
class TimeEstimateCallback(TrainerCallback):
    def __init__(self):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:
            elapsed = time.time() - self.start_time
            steps_done = state.global_step
            if steps_done > 0:
                time_per_step = elapsed / steps_done
                steps_remaining = state.max_steps - steps_done
                eta = timedelta(seconds=int(time_per_step * steps_remaining))
                tqdm.write(
                    f"Step {steps_done}/{state.max_steps}  |  "
                    f"Elapsed: {timedelta(seconds=int(elapsed))}  |  "
                    f"ETA: {eta}"
                )


# ============================================================
# ROBUST DATA LOADER (truncates mismatched lines)
# ============================================================
def load_data_for_pair(pair_name):
    train_dir = os.path.join(DATA_ROOT, "train", pair_name)
    dev_dir   = os.path.join(DATA_ROOT, "dev", pair_name)

    src_lang, tgt_lang = pair_name.split("-")

    with open(os.path.join(train_dir, f"train.{src_lang}"), "r", encoding="utf-8") as f:
        src_lines = [line.strip() for line in f if line.strip()]
    with open(os.path.join(train_dir, f"train.{tgt_lang}"), "r", encoding="utf-8") as f:
        tgt_lines = [line.strip() for line in f if line.strip()]

    # Truncate to the smaller length if they differ
    min_len = min(len(src_lines), len(tgt_lines))
    if len(src_lines) != len(tgt_lines):
        print(f"  ⚠️ Mismatch: src={len(src_lines)}, tgt={len(tgt_lines)}. Truncating to {min_len}.")
        src_lines = src_lines[:min_len]
        tgt_lines = tgt_lines[:min_len]

    assert len(src_lines) == len(tgt_lines), "Mismatch after truncation"
    train_data = [{"en": s, "tgt": t} for s, t in zip(src_lines, tgt_lines)]

    # Dev set (if exists)
    if os.path.exists(dev_dir):
        with open(os.path.join(dev_dir, f"dev.{src_lang}"), "r", encoding="utf-8") as f:
            dev_src = [line.strip() for line in f if line.strip()]
        with open(os.path.join(dev_dir, f"dev.{tgt_lang}"), "r", encoding="utf-8") as f:
            dev_tgt = [line.strip() for line in f if line.strip()]
        min_dev = min(len(dev_src), len(dev_tgt))
        if len(dev_src) != len(dev_tgt):
            print(f"  ⚠️ Dev mismatch: src={len(dev_src)}, tgt={len(dev_tgt)}. Truncating to {min_dev}.")
            dev_src = dev_src[:min_dev]
            dev_tgt = dev_tgt[:min_dev]
        dev_data = [{"en": s, "tgt": t} for s, t in zip(dev_src, dev_tgt)]
    else:
        split = int(len(train_data) * 0.9)
        dev_data = train_data[split:]
        train_data = train_data[:split]

    print(f"  Train samples: {len(train_data):,}  |  Dev samples: {len(dev_data):,}")
    return DatasetDict({
        "train": Dataset.from_list(train_data),
        "validation": Dataset.from_list(dev_data)
    })


# ============================================================
# TOKENIZE
# ============================================================
def preprocess_function(examples, tokenizer, src_lang, tgt_lang):
    inputs = [f"{src_lang} {tgt_lang} {ex}" for ex in examples["en"]]
    targets = [f"{tgt_lang} {ex}" for ex in examples["tgt"]]

    model_inputs = tokenizer(
        inputs, max_length=MAX_LENGTH, truncation=True, padding=False
    )
    labels = tokenizer(
        text_target=targets, max_length=MAX_LENGTH, truncation=True, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# ============================================================
# FINE‑TUNE BODO
# ============================================================
def fine_tune_pair(pair_name):
    print("\n" + "=" * 70)
    print(f"🚀 Fine‑tuning for {pair_name}")
    print("=" * 70)

    dataset = load_data_for_pair(pair_name)

    print("\n📥 Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        use_fast=True
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        torch_dtype=torch.float32
    )
    model.gradient_checkpointing_enable()
    model.to(device)

    print("🔧 Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("\n📝 Tokenizing datasets...")
    src_lang, tgt_lang = pair_name.split("-")
    tokenized_ds = dataset.map(
        lambda x: preprocess_function(x, tokenizer, src_lang, tgt_lang),
        batched=True,
        remove_columns=dataset["train"].column_names,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True
    )

    output_dir = os.path.join(OUTPUT_BASE, pair_name)
    os.makedirs(output_dir, exist_ok=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_steps=SAVE_STEPS,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        predict_with_generate=True,
        generation_max_length=MAX_LENGTH,
        fp16=USE_FP16,
        bf16=USE_BF16,
        push_to_hub=False,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[TimeEstimateCallback()],
    )

    print("\n🏋️ Starting training...\n")
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n✅ Model saved to {output_dir}\n")


# ============================================================
# RUN (ONLY BODO)
# ============================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_BASE, exist_ok=True)

    for pair in LANG_PAIRS:
        fine_tune_pair(pair)

    print("\n🎉 Bodo model fine‑tuned successfully!")
    print("Assamese, Manipuri (Bengali), Manipuri (Meitei Mayek) are already done.")

/home/dingku/jupyter_env/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._regist

Using device: cuda

🚀 Fine‑tuning for eng_Latn-brx_Deva
  ⚠️ Mismatch: src=15235, tgt=15232. Truncating to 15232.
  Train samples: 13,708  |  Dev samples: 1,524

📥 Loading tokenizer and base model...


/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


🔧 Adding LoRA adapters...
trainable params: 3,538,944 || all params: 1,119,082,496 || trainable%: 0.3162362035550952

📝 Tokenizing datasets...


Map:   0%|          | 0/13708 [00:00<?, ? examples/s]

...

Step 4050/5139  |  Elapsed: 1:02:53  |  ETA: 0:16:54
Step 4100/5139  |  Elapsed: 1:03:33  |  ETA: 0:16:06
Step 4150/5139  |  Elapsed: 1:04:14  |  ETA: 0:15:18
Step 4200/5139  |  Elapsed: 1:04:55  |  ETA: 0:14:30
Step 4250/5139  |  Elapsed: 1:05:35  |  ETA: 0:13:43
Step 4300/5139  |  Elapsed: 1:06:17  |  ETA: 0:12:56
Step 4350/5139  |  Elapsed: 1:06:57  |  ETA: 0:12:08
Step 4400/5139  |  Elapsed: 1:07:36  |  ETA: 0:11:21
Step 4450/5139  |  Elapsed: 1:08:15  |  ETA: 0:10:34
Step 4500/5139  |  Elapsed: 1:08:56  |  ETA: 0:09:47


/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step 4550/5139  |  Elapsed: 1:10:10  |  ETA: 0:09:05
Step 4600/5139  |  Elapsed: 1:10:49  |  ETA: 0:08:17
Step 4650/5139  |  Elapsed: 1:11:28  |  ETA: 0:07:31
Step 4700/5139  |  Elapsed: 1:12:07  |  ETA: 0:06:44
Step 4750/5139  |  Elapsed: 1:12:46  |  ETA: 0:05:57
Step 4800/5139  |  Elapsed: 1:13:25  |  ETA: 0:05:11
Step 4850/5139  |  Elapsed: 1:14:04  |  ETA: 0:04:24
Step 4900/5139  |  Elapsed: 1:14:43  |  ETA: 0:03:38
Step 4950/5139  |  Elapsed: 1:15:23  |  ETA: 0:02:52
Step 5000/5139  |  Elapsed: 1:16:03  |  ETA: 0:02:06


/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/dingku/jupyter_env/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step 5050/5139  |  Elapsed: 1:17:16  |  ETA: 0:01:21
Step 5100/5139  |  Elapsed: 1:17:56  |  ETA: 0:00:35

✅ Model saved to /home/dingku/Desktop/FTIT26/eng_Latn-brx_Deva


🎉 Bodo model fine‑tuned successfully!
Assamese, Manipuri (Bengali), Manipuri (Meitei Mayek) are already done.


In [2]:
#Fixed

In [1]:
#!/usr/bin/env python3
import os
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

EXCEL_PATH = "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Meitei/en-mni-Mtei Test.xlsx"
OUTPUT_TXT = "/home/dingku/Desktop/manipuri_meitei_translations.txt"
MODEL_PATH = "/home/dingku/Desktop/FTIT26/eng_Latn-mni_Mtei"
BASE_MODEL = "ai4bharat/indictrans2-en-indic-1B"
COLUMN_NAME = "English Sentences"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 200

print("Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float32
).to(DEVICE)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

src_lang = "eng_Latn"
tgt_lang = "mni_Mtei"

df = pd.read_excel(EXCEL_PATH)
sentences = df[COLUMN_NAME].dropna().astype(str).tolist()
print(f"Found {len(sentences)} sentences to translate.")

translations = []

for i, sent in enumerate(tqdm(sentences, desc="Translating"), start=1):
    if not sent.strip():
        translations.append("")
        continue

    input_text = f"{src_lang} {tgt_lang} {sent}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            repetition_penalty=2.0,
            length_penalty=1.0,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if translation.startswith(tgt_lang):
        translation = translation[len(tgt_lang):].strip()

    translations.append(translation)
    print(f"\n[{i}/{len(sentences)}]\nEN: {sent}\nMT: {translation}\n" + "-"*50)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    for trans in translations:
        f.write(trans + "\n")

print(f"\n✅ Translations saved to: {OUTPUT_TXT}")

/home/dingku/jupyter_env/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new downlo

Loading base model and tokenizer...
Loading LoRA adapter...
Found 1000 sentences to translate.


Translating:   0%|                             | 1/1000 [00:01<25:50,  1.55s/it]


[1/1000]
EN: NDTV has learnt from sources that the T20 World Cup 2026 match between India and Pakistan is ON.
MT: ꯑꯦꯟꯗꯤꯇꯤꯚꯥꯏꯅꯥ ꯁꯣꯔꯁꯤꯡꯗꯒꯤ ꯈꯪꯂꯦ ꯃꯗꯨꯗꯤ ꯚꯥꯔꯠ ꯑꯃꯁꯨꯡ ꯄꯥꯀꯤꯁꯇꯥꯟꯒꯤ ꯃꯔꯛꯇ ꯇꯤ20 ꯋꯥꯔꯂ ꯭ ꯗ ꯀꯞ 2026ꯒꯤ ꯃꯦꯆ ꯑꯗꯨ ꯑꯣꯅ
--------------------------------------------------


Translating:   0%|                             | 2/1000 [00:02<15:14,  1.09it/s]

...


[998/1000]
EN: Oncologist shares case of non-tobacco related cancer in 21-year-old man with ‘sharp teeth,’ says ‘It could have been identified’
MT: ꯑꯣꯡꯀꯣꯂꯣꯖꯤꯁꯇꯅꯥ ꯆꯍꯤ 21 ꯁꯨꯔꯕꯥ ꯅꯨꯄꯥꯗꯥ "ꯄꯥꯔꯄ ꯗꯊꯁꯤꯡ ꯂꯩꯕ ꯅꯟ-ꯇꯣꯕꯦꯀꯣ ꯔꯤꯂꯦꯃꯦꯇ ꯀꯦꯟꯁꯔꯒꯤ ꯀꯦꯁ ꯁꯦꯌꯔ ꯇꯧꯔꯦ, ꯍꯥꯏꯔꯤ  ꯃꯁꯤ ꯃꯁꯛ ꯈꯪꯗꯣꯛꯄꯥ ꯉꯝꯒꯅꯤ ꯫
--------------------------------------------------


Translating: 100%|██████████████████████████▉| 999/1000 [14:56<00:01,  1.04s/it]


[999/1000]
EN: Nutritionist says pineapple and cinnamon may help reduce period cramps naturally
MT: ꯅꯇꯁꯤꯁꯇꯅꯥ ꯍꯥꯏꯔꯤ ꯃꯗꯨꯗꯤ ꯑꯅꯤꯄꯂ ꯑꯃꯁꯨꯡ ꯁꯤꯅꯥꯃꯣꯅꯥ ꯃꯍꯧꯁꯥꯒꯤ ꯑꯣꯏꯕ ꯃꯑꯣꯡꯗ ꯄꯤꯔꯨꯗ ꯀ ꯭ ꯔꯦꯝꯄꯁꯤꯡ ꯍꯟꯊꯍꯟꯕꯗꯥ ꯃꯇꯦꯡ ꯄꯥꯡꯕ ꯌꯥꯏ ꯫
--------------------------------------------------


Translating: 100%|██████████████████████████| 1000/1000 [14:57<00:00,  1.11it/s]


[1000/1000]
EN: Don’t brush before bed? Your heart might pay the price
MT: ꯅꯨꯡꯊꯤꯜꯒꯤ ꯃꯃꯥꯡꯗ ꯕꯁ ꯇꯧꯔꯣꯏꯗꯕꯅꯤ ꯫ ꯅꯍꯥꯛꯀꯤ ꯊꯝꯃꯣꯏꯅ ꯃꯃꯜ ꯊꯤꯕꯥ ꯌꯥꯏ ꯫
--------------------------------------------------

✅ Translations saved to: /home/dingku/Desktop/manipuri_meitei_translations.txt


In [2]:
#!/usr/bin/env python3

#Finetuned IndicTrans2 transalating the test data.

import os
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================
LANG_CONFIG = {
    "asm": {
        "excel": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Assamese/en-as Test.xlsx",
        "output": "/home/dingku/Desktop/assamese_translations.txt",
        "model": "/home/dingku/Desktop/FTIT26/eng_Latn-asm_Beng",
        "tgt_tag": "asm_Beng"
    },
    "mni_beng": {
        "excel": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Manipuri/en-mni Test.xlsx",
        "output": "/home/dingku/Desktop/manipuri_bengali_translations.txt",
        "model": "/home/dingku/Desktop/FTIT26/eng_Latn-mni_Beng",
        "tgt_tag": "mni_Beng"
    },
    "mni_mtei": {
        "excel": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 1/English - Meitei/en-mni-Mtei Test.xlsx",
        "output": "/home/dingku/Desktop/manipuri_meitei_translations.txt",
        "model": "/home/dingku/Desktop/FTIT26/eng_Latn-mni_Mtei",
        "tgt_tag": "mni_Mtei"
    },
    "brx": {
        "excel": "/home/dingku/Desktop/WMT 2026 IndicMT Test Data/Category 2/English - Bodo/en-bodo Test.xlsx",
        "output": "/home/dingku/Desktop/bodo_translations.txt",
        "model": "/home/dingku/Desktop/FTIT26/eng_Latn-brx_Deva",
        "tgt_tag": "brx_Deva"
    }
}

BASE_MODEL = "ai4bharat/indictrans2-en-indic-1B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLUMN_NAME = "English Sentences"
MAX_NEW_TOKENS = 200

# Generation parameters that work for Meitei Mayek
GEN_KWARGS = {
    "num_beams": 4,
    "early_stopping": True,
    "no_repeat_ngram_size": 2,
    "repetition_penalty": 2.0,
    "length_penalty": 1.0,
    "do_sample": False,
    "temperature": 1.0,
    "pad_token_id": None  # will be set later
}

# ============================================================
# LOAD MODEL & TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float32
).to(DEVICE)

# ============================================================
# TRANSLATE FUNCTION
# ============================================================
def translate(lang_code, config):
    print(f"\n{'='*60}\nTranslating {lang_code}...\n{'='*60}")
    
    # Load adapter
    model = PeftModel.from_pretrained(base_model, config["model"])
    model.eval()
    
    # Optionally merge (uncomment if needed)
    # model = model.merge_and_unload()
    
    # Read Excel
    df = pd.read_excel(config["excel"])
    sentences = df[COLUMN_NAME].dropna().astype(str).tolist()
    print(f"Found {len(sentences)} sentences.")
    
    src_tag = "eng_Latn"
    tgt_tag = config["tgt_tag"]
    
    translations = []
    for i, sent in enumerate(tqdm(sentences, desc="Translating"), start=1):
        if not sent.strip():
            translations.append("")
            continue
        
        input_text = f"{src_tag} {tgt_tag} {sent}"
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)
        
        gen_kwargs = GEN_KWARGS.copy()
        gen_kwargs["pad_token_id"] = tokenizer.eos_token_id
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                **gen_kwargs
            )
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Remove any language tag that might appear at start
        if translation.startswith(tgt_tag):
            translation = translation[len(tgt_tag):].strip()
        
        translations.append(translation)
        print(f"\n[{i}/{len(sentences)}]\nEN: {sent}\n{lang_code.upper()}: {translation}\n" + "-"*50)
    
    # Save
    with open(config["output"], "w", encoding="utf-8") as f:
        for trans in translations:
            f.write(trans + "\n")
    print(f"✅ Saved to {config['output']}")

# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    for lang in ["asm", "mni_beng", "mni_mtei", "brx"]:
        translate(lang, LANG_CONFIG[lang])
    print("\n🎉 All translations attempted.")

/home/dingku/jupyter_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Translating asm...
Found 1000 sentences.


Translating:   0%|                             | 1/1000 [00:01<26:13,  1.57s/it]


[1/1000]
EN: NDTV has learnt from sources that the T20 World Cup 2026 match between India and Pakistan is ON.
ASM: . দেৱাৰ বির্জীৰ ৰালে, বাৰে মি জিৰি 5 2026 а : ?
--------------------------------------------------


Translating:   0%|                             | 2/1000 [00:02<24:37,  1.48s/it]

...


[998/1000]
EN: Oncologist shares case of non-tobacco related cancer in 21-year-old man with ‘sharp teeth,’ says ‘It could have been identified’
BRX: केनसारारि फाहामगिरि आरो देहा फाहामगिरिफोरा बुङो दि बे बेरामखौ होबथानो थाखाय मुलिनि बाहायनायखौ खम खालामनाय जादों |
--------------------------------------------------


Translating: 100%|██████████████████████████▉| 999/1000 [14:21<00:00,  1.05it/s]


[999/1000]
EN: Nutritionist says pineapple and cinnamon may help reduce period cramps naturally
BRX: नीउथ्रिसनिस्टआ बुङो दि आनारस आरो दालसिनिआ माहाजोननि खिन्थिनायखौ मिथिंगायारियै बाङाइ खालामनायाव मदद होनो हागौ |
--------------------------------------------------


Translating: 100%|██████████████████████████| 1000/1000 [14:21<00:00,  1.16it/s]


[1000/1000]
EN: Don’t brush before bed? Your heart might pay the price
BRX: उन्दुनायनि सिगां ब्रास दाखालामनो नाङा > नोंथांनि बिखाया बेसेन होनो हागौ |
--------------------------------------------------
✅ Saved to /home/dingku/Desktop/bodo_translations.txt

🎉 All translations attempted.
